# Day 1 上午：从文本到向量（3h）

## 学习目标

1. 理解神经网络训练循环 (Training Loop) 的四个步骤
2. 掌握分词器 (Tokenizer) 的工作原理
3. 理解嵌入 (Embedding) 的本质：查表 + 语义空间
4. 学会用提示工程 (Prompt Engineering) 解决实际任务
5. 建立企业决策框架：何时微调 / RAG / Prompt

---

## 三天课程总览

| 天 | 上午 | 下午 |
|:---:|------|------|
| **Day 1** | **从文本到向量**（训练循环 → Tokenizer → Embedding → Prompt） | Transformer 架构与注意力机制 |
| **Day 2** | SFT 与 LoRA 微调实战 | 预训练与对齐优化 (RLHF/DPO) |
| **Day 3** | 评测选型与 Agent/RAG | RAG 实战与工程桥接 |

## 企业 AI 全景：大模型生命周期

```
预训练 (Pretrain) → 监督微调 (SFT) → 对齐 (RLHF/DPO) → 部署 (Deploy)
     ↑                    ↑                  ↑                ↑
  海量数据            标注数据           人类偏好          推理优化
  数百万$            数千~万$           数千$             持续成本
```

> **Day 3 你将构建的东西：** 一个完整的 RAG + Agent 系统，能够对接企业知识库回答问题。
>
> 但首先，让我们从最基础的开始 —— 神经网络是怎么训练的？

In [ ]:
# ============================================================
# 环境配置
# ============================================================
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), "utils"))
sys.path.insert(0, "utils")

# 加载 .env 配置（Day0 中保存的 API Key 和模型选择）
from config import setup
env = setup()

In [ ]:
# 基础依赖
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

# 中文字体配置
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK SC", "Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11

# 随机种子
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("环境准备完成！")

---
## 第一部分：训练循环 (Training Loop)（30 min）

### 核心问题：神经网络是怎么学习的？

答案：四步循环，反复迭代。

```
┌─────────────────────────────────────┐
│  Forward → Loss → Backward → Step  │
│  前向传播   损失    反向传播   更新   │
│         不断重复这四步              │
└─────────────────────────────────────┘
```

我们用一个实际的二分类问题来演示。

### 1.1 创建数据集

我们用 sklearn 生成一个**月亮形数据集** —— 两类数据点交错排列，无法用直线分开。

In [ ]:
# 生成月亮形数据集
n_samples = 1000
X, y = make_moons(n_samples=n_samples, noise=0.1, random_state=42)

# 转换为 PyTorch 张量 (Tensor)
X_tensor = torch.FloatTensor(X)
y_tensor = torch.FloatTensor(y).reshape(-1, 1)

print(f"数据形状: X={X_tensor.shape}, y={y_tensor.shape}")
print(f"特征维度: {X_tensor.shape[1]} (x坐标, y坐标)")

# 可视化
plt.figure(figsize=(8, 5))
plt.scatter(X[y==0, 0], X[y==0, 1], c='royalblue', label='类别 0', alpha=0.6, s=15)
plt.scatter(X[y==1, 0], X[y==1, 1], c='tomato', label='类别 1', alpha=0.6, s=15)
plt.legend()
plt.title('月亮形数据集 — 线性不可分')
plt.xlabel('x1')
plt.ylabel('x2')
plt.tight_layout()
plt.show()

### 1.2 搭建神经网络

我们用一个简单的多层感知机 (MLP)：

```
输入 (2维) → 隐藏层1 (32) → ReLU → 隐藏层2 (16) → ReLU → 输出 (1) → Sigmoid
```

In [ ]:
# 定义神经网络
model = nn.Sequential(
    nn.Linear(2, 32),    # 输入层 -> 隐藏层1
    nn.ReLU(),           # 激活函数: max(0, x)
    nn.Linear(32, 16),   # 隐藏层1 -> 隐藏层2
    nn.ReLU(),
    nn.Linear(16, 1),    # 隐藏层2 -> 输出层
    nn.Sigmoid()         # 输出概率 (0~1)
)

print("模型结构:")
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"\n总参数量: {total_params}")
print(f"\n企业视角: GPT-4 有约 1.8 万亿参数，我们这个只有 {total_params} 个，但训练原理完全一样！")

### 1.3 训练循环四步曲

| 步骤 | 代码 | 含义 |
|------|------|------|
| **Forward** | `y_pred = model(X)` | 数据过一遍网络，得到预测 |
| **Loss** | `loss = criterion(y_pred, y)` | 计算预测与真实值的差距 |
| **Backward** | `loss.backward()` | 计算每个参数的梯度（该往哪个方向调） |
| **Step** | `optimizer.step()` | 根据梯度更新参数 |

In [ ]:
# 定义损失函数和优化器
criterion = nn.BCELoss()  # 二分类交叉熵损失 (Binary Cross Entropy)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  # Adam 优化器

print("损失函数: BCELoss（衡量预测概率与真实标签的差距）")
print("优化器: Adam（自适应学习率，企业中最常用）")
print("学习率: 0.01（每次参数更新的步长）")

## 🏋️ 练习 1：补全训练循环四步曲（10分钟）⭐

**目标**：在下方代码中填写训练循环的四个关键步骤

**步骤**：
1. Forward：用模型对 `X_tensor` 做前向传播，得到预测值
2. Loss：用 `criterion` 计算预测值与真实值 `y_tensor` 的损失
3. Backward：先清空梯度 `optimizer.zero_grad()`，再反向传播 `loss.backward()`
4. Step：用优化器更新参数 `optimizer.step()`

**预期输出**：
```
Epoch 100 | Loss: 0.0xxx | Accuracy: 1.0000
...
✅ 验证通过！最终准确率: 1.0000
```

<details>
<summary>💡 提示1：思路方向</summary>

四步分别对应：`model(输入)`, `criterion(预测, 真实)`, `loss.backward()`, `optimizer.step()`

</details>

<details>
<summary>💡 提示2：关键代码</summary>

```python
y_pred = model(数据)
loss = criterion(y_pred, 标签)
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

</details>

<details>
<summary>💡 提示3：参考答案</summary>

```python
outputs = model(X_tensor)
loss = criterion(outputs, y_tensor)
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

</details>

In [ ]:
# 练习 1：补全训练循环四步曲
import time

epochs = 500
losses = []

print("开始训练...")
print("=" * 50)
start_time = time.time()

for epoch in range(epochs):
    # ↓↓↓ 你的代码（约4行）↓↓↓
    outputs = model(X_tensor)
    loss = criterion(outputs, y_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # ↑↑↑ 你的代码 ↑↑↑

    losses.append(loss.item())

    if (epoch + 1) % 100 == 0:
        acc = ((outputs > 0.5).float() == y_tensor).float().mean()
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f} | Accuracy: {acc:.4f}")

elapsed = time.time() - start_time
print("=" * 50)
print(f"训练完成！耗时: {elapsed:.2f} 秒")

# --------- 验证 ---------
assert loss.item() < 0.1, f"Loss 应小于 0.1，当前为 {loss.item():.4f}"
with torch.no_grad():
    final_acc = ((model(X_tensor) > 0.5).float() == y_tensor).float().mean()
assert final_acc > 0.9, f"准确率应大于 90%，当前为 {final_acc:.4f}"
print(f"\n✅ 验证通过！最终准确率: {final_acc:.4f}")

In [ ]:
# 可视化训练过程
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：Loss 下降曲线
axes[0].plot(losses, color='blue', linewidth=1, alpha=0.3, label='原始损失')
window = 20
smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
axes[0].plot(range(window-1, len(losses)), smoothed, color='blue', linewidth=2, label='平滑曲线')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('训练损失下降曲线')
axes[0].legend()

# 右图：决策边界
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
grid = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()])

with torch.no_grad():
    Z = model(grid).numpy().reshape(xx.shape)

axes[1].contourf(xx, yy, Z, levels=50, cmap='RdBu_r', alpha=0.6)
axes[1].scatter(X[y==0, 0], X[y==0, 1], c='royalblue', s=10, alpha=0.5)
axes[1].scatter(X[y==1, 0], X[y==1, 1], c='tomato', s=10, alpha=0.5)
axes[1].set_title('学习到的决策边界')

plt.tight_layout()
plt.show()

---
## 第二部分：Tokenizer — 文本如何变数字（20 min）

### 核心问题：模型只认数字，文本怎么办？

```
"你好世界" → Tokenizer → [57668, 53901, ...] → 模型 → [1312, ...] → "Hello"
```

### 分词策略对比

| 策略 | 示例 | 优缺点 |
|------|------|--------|
| 字符级 | `["H","e","l","l","o"]` | 词表小，但序列太长 |
| 单词级 | `["Hello", "world"]` | 直观，但词表太大、不能处理新词 |
| **子词级 (BPE)** | `["Hello", ",", "world"]` | 平衡！GPT/LLaMA/Qwen 都用 |

In [ ]:
# 演示不同分词策略
text = "Hello, I'm learning machine learning!"

# 1. 字符级
char_tokens = list(text)
print(f"字符级分词 ({len(char_tokens)} tokens): {char_tokens[:10]}...")

# 2. 单词级
word_tokens = text.split()
print(f"单词级分词 ({len(word_tokens)} tokens): {word_tokens}")

# 3. 子词级 (BPE) - 使用 tiktoken
try:
    import tiktoken
    enc = tiktoken.get_encoding("cl100k_base")  # GPT-4 的编码器
    bpe_tokens = enc.encode(text)
    print(f"BPE 分词 ({len(bpe_tokens)} tokens): {bpe_tokens}")
    print(f"\n解码每个 token:")
    for t in bpe_tokens:
        print(f"  {t}: '{enc.decode([t])}'")
except ImportError:
    print("\n请安装 tiktoken: pip install tiktoken")
    print("BPE 分词示例: [9906, 11, 358, 2846, 6975, 5780, 6975, 0]")

In [ ]:
# tiktoken 详细演示：看看中文是怎么切分的
try:
    import tiktoken
    enc = tiktoken.get_encoding("cl100k_base")
    
    text_cn = "你好！我正在学习大语言模型。"
    tokens = enc.encode(text_cn)
    
    print(f"原文: {text_cn}")
    print(f"Token IDs ({len(tokens)} tokens): {tokens}")
    print(f"\n逐 token 解码:")
    for t in tokens:
        decoded = enc.decode([t])
        print(f"  ID {t:>6d} → '{decoded}'")
    
    print(f"\n注意: {len(text_cn)} 个中文字符 → {len(tokens)} 个 token")
    print(f"中文平均每个字符需要 {len(tokens)/len(text_cn):.1f} 个 token")
    
except ImportError:
    print("请安装 tiktoken: pip install tiktoken")

In [ ]:
# 不同语言的 Token 效率对比
try:
    import tiktoken
    enc = tiktoken.get_encoding("cl100k_base")
    
    texts = {
        "English": "Machine learning is a branch of artificial intelligence.",
        "Chinese": "机器学习是人工智能的一个分支。",
        "Code": "def hello(): return 'Hello, World!'",
        "Mixed": "GPT-4的参数量约为1.8万亿",
    }
    
    print("不同语言的 Token 效率:")
    print("=" * 60)
    
    for lang, text in texts.items():
        tokens = enc.encode(text)
        ratio = len(text) / len(tokens)
        print(f"\n{lang}:")
        print(f"  文本: {text}")
        print(f"  字符数: {len(text)} | Token 数: {len(tokens)}")
        print(f"  效率: {ratio:.2f} 字符/token")
    
    print("\n" + "=" * 60)
    print("企业启示: 中文 token 效率低 → API 调用成本更高!")
    print("选择中文优化的模型（如 Qwen）可以降低 token 消耗。")
    
except ImportError:
    print("请安装 tiktoken: pip install tiktoken")
    print("\n预期结果:")
    print("English: ~5.2 字符/token")
    print("Chinese: ~1.5 字符/token (效率较低，成本更高)")
    print("Code:    ~3.8 字符/token")

---
## 🏋️ 练习 2：企业Token成本计算器（15分钟）⭐⭐

**目标**：编写函数估算不同分词器下的API调用成本，对比中文优化方案的省钱效果

**场景**：你的公司每天处理1万次客户查询，每次约200个中文字符。API价格为 ¥0.06/千tokens（输入）。

**步骤**：
1. 编写 `estimate_daily_cost(text_samples, price_per_1k)` 函数：对样本编码、计算平均token数、推算日/月成本
2. 编写 `estimate_daily_cost_cn_optimized()` 函数：模拟中文优化分词器（每字符约0.6个token）
3. 打印对比表格，回答"切换中文优化模型每月省多少钱？"

**预期输出**：
```
分词器         日Token量     日成本(¥)     月成本(¥)
cl100k_base   338,000       20.28        608.40
中文优化       196,800       11.81        354.24
月节省                                    254.16
```

<details>
<summary>💡 提示1：思路方向</summary>

用 `enc.encode(text)` 获取token列表，`len()` 获取数量。日token量 = 平均token数 × 日查询量。

</details>

<details>
<summary>💡 提示2：关键代码</summary>

```python
token_counts = [len(enc.encode(t)) for t in text_samples]
avg_tokens = sum(token_counts) / len(token_counts)
daily_tokens = avg_tokens * DAILY_QUERIES
daily_cost = daily_tokens / 1000 * price_per_1k
```

</details>

<details>
<summary>💡 提示3：参考答案</summary>

```python
def estimate_daily_cost(text_samples, price_per_1k, encoder_name="cl100k_base"):
    enc = tiktoken.get_encoding(encoder_name)
    token_counts = [len(enc.encode(t)) for t in text_samples]
    avg_tokens = sum(token_counts) / len(token_counts)
    daily_tokens = avg_tokens * DAILY_QUERIES
    daily_cost = daily_tokens / 1000 * price_per_1k
    monthly_cost = daily_cost * 30
    return {"avg_tokens": avg_tokens, "daily_tokens": daily_tokens,
            "daily_cost": daily_cost, "monthly_cost": monthly_cost}
```

</details>

In [ ]:
# 练习 2：企业Token成本计算器
import tiktoken

# 企业场景参数
DAILY_QUERIES = 10000
AVG_CHARS_PER_QUERY = 200
PRICE_PER_1K_TOKENS = 0.06  # ¥/千tokens

# 模拟企业查询样本
sample_queries = [
    "你好，我想查询一下我的订单状态，订单号是2024031401，请帮我看一下物流信息",
    "我购买的商品出现了质量问题，想申请退货退款，应该怎么操作？",
    "请问你们的会员积分兑换规则是什么？我有5000积分可以换什么？",
    "我的账户密码忘记了，尝试了好几次都登录不了，能帮我重置一下吗？",
    "你们最近有什么优惠活动吗？我想买一台笔记本电脑，有推荐的型号吗？",
]

# 步骤1：编写成本估算函数
def estimate_daily_cost(text_samples, price_per_1k, encoder_name="cl100k_base"):
    """估算日均Token成本，返回 dict: {avg_tokens, daily_tokens, daily_cost, monthly_cost}"""
    enc = tiktoken.get_encoding(encoder_name)
    # ↓↓↓ 你的代码（约5行）↓↓↓
    token_counts = [len(enc.encode(t)) for t in text_samples]
    avg_tokens = sum(token_counts) / len(token_counts)
    daily_tokens = avg_tokens * DAILY_QUERIES
    daily_cost = daily_tokens / 1000 * price_per_1k
    monthly_cost = daily_cost * 30
    # ↑↑↑ 你的代码 ↑↑↑
    return {"avg_tokens": avg_tokens, "daily_tokens": daily_tokens,
            "daily_cost": daily_cost, "monthly_cost": monthly_cost}

# 步骤2：模拟中文优化分词器（每字符约0.6个token）
def estimate_daily_cost_cn_optimized(text_samples, price_per_1k):
    """中文优化分词器：假设每个中文字符平均只需0.6个token"""
    # ↓↓↓ 你的代码（约5行）↓↓↓
    avg_chars = sum(len(t) for t in text_samples) / len(text_samples)
    avg_tokens = avg_chars * 0.6
    daily_tokens = avg_tokens * DAILY_QUERIES
    daily_cost = daily_tokens / 1000 * price_per_1k
    monthly_cost = daily_cost * 30
    # ↑↑↑ 你的代码 ↑↑↑
    return {"avg_tokens": avg_tokens, "daily_tokens": daily_tokens,
            "daily_cost": daily_cost, "monthly_cost": monthly_cost}

# 步骤3：对比两种方案
result_cl100k = estimate_daily_cost(sample_queries, PRICE_PER_1K_TOKENS)
result_cn = estimate_daily_cost_cn_optimized(sample_queries, PRICE_PER_1K_TOKENS)

# 步骤4：打印对比表格
print("=" * 60)
print(f"{'分词器':<12} {'日Token量':>10} {'日成本(¥)':>10} {'月成本(¥)':>10}")
print("-" * 60)
print(f"{'cl100k_base':<12} {result_cl100k['daily_tokens']:>10,.0f} {result_cl100k['daily_cost']:>10.2f} {result_cl100k['monthly_cost']:>10.2f}")
print(f"{'中文优化':<12} {result_cn['daily_tokens']:>10,.0f} {result_cn['daily_cost']:>10.2f} {result_cn['monthly_cost']:>10.2f}")
savings = result_cl100k['monthly_cost'] - result_cn['monthly_cost']
print("-" * 60)
print(f"{'月节省':<12} {'':>10} {'':>10} {savings:>10.2f}")
print("=" * 60)
print(f"\n切换中文优化模型每月可节省: ¥{savings:.2f}")

# --------- 验证 ---------
assert result_cl100k is not None, "estimate_daily_cost 返回了 None"
assert result_cn is not None, "estimate_daily_cost_cn_optimized 返回了 None"
assert result_cl100k['daily_tokens'] > 0, "日Token量应大于0"
assert result_cl100k['monthly_cost'] > result_cn['monthly_cost'], "中文优化应该更便宜"
print("\n✅ 所有验证通过！")

---
## 休息（10 min）

---
## 第三部分：Embedding — 查表 + 语义空间（20 min）

### 核心问题：Token ID 只是编号，怎么表达语义？

答案：给每个 token 分配一个**向量** (Embedding)，让相似的词有相似的向量。

```
"猫" → token_id: 123 → 查 Embedding 表 → [0.2, -0.1, 0.8, ...] (512维向量)
"狗" → token_id: 456 → 查 Embedding 表 → [0.3, -0.2, 0.7, ...] (相似的向量！)
```

`nn.Embedding` 本质就是一个**可学习的查表矩阵**，形状为 `[词表大小, 向量维度]`。

In [ ]:
# Embedding 的本质：查表
vocab_size = 10   # 词表大小
embed_dim = 4     # 每个词用 4 维向量表示

embedding = nn.Embedding(vocab_size, embed_dim)

print("Embedding 矩阵形状:", embedding.weight.shape)
print(f"→ {vocab_size} 个词，每个用 {embed_dim} 维向量表示")
print("\nEmbedding 矩阵内容（随机初始化）:")
print(embedding.weight.data)

In [ ]:
# Embedding 就是查表！
word_idx = torch.tensor([0, 3, 5])  # 要查的词索引

# 方法1: 使用 Embedding 层
result1 = embedding(word_idx)

# 方法2: 直接索引矩阵（完全等价！）
result2 = embedding.weight[word_idx]

print("查询词索引:", word_idx.tolist())
print("\n方法1 - embedding(idx):")
print(result1.data)
print("\n方法2 - weight[idx]:")
print(result2.data)
print("\n两种方法完全等价:", torch.allclose(result1, result2))
print("\n结论: nn.Embedding 本质就是矩阵索引，没有任何复杂计算！")

### 3.2 训练词向量：通过预测下一个词来学习语义

Embedding 矩阵的值是通过训练学到的。训练目标：**预测下一个词**。

```
"机器" → 模型 → 预测 "学习"（高概率）
"深度" → 模型 → 预测 "学习"（高概率）
```

因为 "机器" 和 "深度" 经常预测同样的下一个词，训练后它们的向量会很相似。

In [ ]:
# 准备训练数据
text = """
机器学习是人工智能的核心
深度学习是机器学习的分支
神经网络是深度学习的基础
人工智能改变世界
机器学习改变生活
"""

# 按字符分词
words = list(text.replace('\n', '').replace(' ', ''))
vocab = sorted(set(words))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
vocab_size = len(vocab)

print(f"词表大小: {vocab_size}")
print(f"词表: {vocab}")

# 构建 (当前词, 下一个词) 训练对
X_emb, Y_emb = [], []
for i in range(len(words) - 1):
    X_emb.append(word2idx[words[i]])
    Y_emb.append(word2idx[words[i+1]])

X_emb = torch.tensor(X_emb)
Y_emb = torch.tensor(Y_emb)
print(f"\n训练样本数: {len(X_emb)}")
print(f"前5个样本:")
for i in range(5):
    print(f"  '{idx2word[X_emb[i].item()]}' → '{idx2word[Y_emb[i].item()]}'")

In [ ]:
# Bigram 语言模型：预测下一个词
class BigramModel(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, vocab_size)
    
    def forward(self, x):
        emb = self.embedding(x)       # [batch] → [batch, embed_dim]
        logits = self.linear(emb)     # [batch, embed_dim] → [batch, vocab_size]
        return logits

embed_dim = 16
bigram_model = BigramModel(vocab_size, embed_dim)
print(f"模型参数量: {sum(p.numel() for p in bigram_model.parameters())}")

In [ ]:
# 训练 Bigram 模型
optimizer = torch.optim.Adam(bigram_model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

losses = []
for epoch in range(500):
    logits = bigram_model(X_emb)
    loss = criterion(logits, Y_emb)
    losses.append(loss.item())
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f}")

print(f"\n训练完成! 最终 Loss: {losses[-1]:.4f}")

In [ ]:
# 可视化学到的词向量
from sklearn.decomposition import PCA

word_vectors = bigram_model.embedding.weight.detach().numpy()

# 用 PCA 降到 2 维可视化
pca = PCA(n_components=2)
word_vectors_2d = pca.fit_transform(word_vectors)

plt.figure(figsize=(10, 8))
plt.scatter(word_vectors_2d[:, 0], word_vectors_2d[:, 1], c='steelblue', s=100, zorder=5)

for i, word in enumerate(vocab):
    plt.annotate(word, (word_vectors_2d[i, 0], word_vectors_2d[i, 1]),
                fontsize=14, ha='center', va='bottom',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.title('训练后的词向量空间（PCA 降维）')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("观察: 语义相关的词（如'机'和'器'，'学'和'习'）在空间中更接近。")

---
## 🏋️ 练习 3：实现余弦相似度（10分钟）⭐

**目标**：实现余弦相似度函数，并用它查找词向量空间中最相似的词

**公式**：

$$\text{cos\_sim}(a, b) = \frac{a \cdot b}{|a| \times |b|}$$

- 返回 1：方向完全相同
- 返回 0：正交（无关）
- 返回 -1：方向完全相反

**步骤**：
1. 补全 `cosine_similarity(v1, v2)` 函数（计算点积、范数、返回商）
2. 通过基础验证用例
3. 用该函数找出词表中与"学"和"机"最相似的词

**预期输出**：
```
cos_sim(v, v) = 1.0000  (应为 1.0)
cos_sim(v, -v) = -1.0000  (应为 -1.0)
✅ 基础验证通过！

与 '学' 最相似的词:
  习: 0.xxxx
  ...
```

<details>
<summary>💡 提示1：思路方向</summary>

需要用到 `np.dot()` 计算点积，`np.linalg.norm()` 计算向量长度

</details>

<details>
<summary>💡 提示2：关键代码</summary>

```python
dot_product = np.dot(v1, v2)
norm1 = np.linalg.norm(v1)
norm2 = np.linalg.norm(v2)
similarity = dot_product / (norm1 * norm2)
```

</details>

<details>
<summary>💡 提示3：参考答案</summary>

```python
def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
```

</details>

In [ ]:
# 练习 3：实现余弦相似度

def cosine_similarity(v1, v2):
    """计算两个向量的余弦相似度"""
    # ↓↓↓ 你的代码（约3行）↓↓↓
    dot_product = np.dot(v1, v2)
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    return dot_product / (norm1 * norm2)
    # ↑↑↑ 你的代码 ↑↑↑

# --------- 验证 ---------
test_vec = np.array([1.0, 2.0, 3.0])

sim_same = cosine_similarity(test_vec, test_vec)
sim_opposite = cosine_similarity(test_vec, -test_vec)

print(f"cos_sim(v, v) = {sim_same:.4f}  (应为 1.0)")
print(f"cos_sim(v, -v) = {sim_opposite:.4f}  (应为 -1.0)")

assert abs(sim_same - 1.0) < 1e-6, f"cos_sim(v, v) 应为 1.0，得到 {sim_same}"
assert abs(sim_opposite - (-1.0)) < 1e-6, f"cos_sim(v, -v) 应为 -1.0，得到 {sim_opposite}"
print("✅ 基础验证通过！")

# 用你实现的函数找相似词
def find_similar(word, top_k=5):
    if word not in word2idx:
        print(f"词 '{word}' 不在词表中")
        return
    word_vec = word_vectors[word2idx[word]]
    similarities = []
    for w in vocab:
        if w != word:
            sim = cosine_similarity(word_vec, word_vectors[word2idx[w]])
            similarities.append((w, sim))
    similarities.sort(key=lambda x: x[1], reverse=True)
    print(f"\n与 '{word}' 最相似的词:")
    for w, sim in similarities[:top_k]:
        print(f"  {w}: {sim:.4f}")

find_similar('学')
find_similar('机')

---
## 🏋️ 练习 4：用预训练 Embedding 找语义相似句（10分钟）⭐⭐

**目标**：使用预训练 Embedding 模型计算句子间的语义相似度，并按相似度排序

刚才我们训练了一个非常小的词向量。实际企业中，会使用**预训练好的 Embedding 模型**，如阿里百炼的 `text-embedding-v3`。

**步骤**：
1. 用 `env.get_embedder()` 获取 Embedding 后端
2. 用 `embedder.embed(sentences)` 获取句子向量
3. 用练习3实现的 `cosine_similarity` 计算查询句与每个句子的相似度，按高到低排序
4. （加分）用 KMeans 对句子聚类

**预期输出**：
```
查询: 'AI 技术帮助企业提高效率'

相似度排名:
  0.xxxx | 机器学习可以分析大量数据
  0.xxxx | 数据挖掘帮助企业做决策
  ...
```

> 注意：此练习需要已配置 API Key，请确保已运行 Day0 环境配置。

<details>
<summary>💡 提示1：思路方向</summary>

用 `embedder.embed(sentences)` 获取向量列表，用你实现的 `cosine_similarity` 计算相似度

</details>

<details>
<summary>💡 提示2：关键代码</summary>

```python
vectors = embedder.embed(sentences)
query_vec = embedder.embed([query])[0]
for i, sent in enumerate(sentences):
    sim = cosine_similarity(np.array(query_vec), np.array(vectors[i]))
```

</details>

<details>
<summary>💡 提示3：参考答案（含聚类加分）</summary>

```python
vectors = embedder.embed(sentences)
query_vec = embedder.embed([query])[0]
results = []
for i, sent in enumerate(sentences):
    sim = cosine_similarity(np.array(query_vec), np.array(vectors[i]))
    results.append((sim, sent))
results.sort(reverse=True)

# 聚类加分
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=2, random_state=42)
labels = kmeans.fit_predict(np.array(vectors))
```

</details>

In [ ]:
# 练习 4：用预训练 Embedding 找语义相似句

# 获取 Embedding 后端
embedder = env.get_embedder()

# 准备句子
sentences = [
    "机器学习可以分析大量数据",
    "深度学习是人工智能的重要分支",
    "今天的天气非常好",
    "数据挖掘帮助企业做决策",
    "公园里的花开了",
]

query = "AI 技术帮助企业提高效率"

# ↓↓↓ 你的代码（约8行）↓↓↓
# 获取向量
vectors = embedder.embed(sentences)
query_vec = embedder.embed([query])[0]

# 计算相似度并排序
results = []
for i, sent in enumerate(sentences):
    sim = cosine_similarity(np.array(query_vec), np.array(vectors[i]))
    results.append((sim, sent))
results.sort(reverse=True)
# ↑↑↑ 你的代码 ↑↑↑

# 打印结果
print(f"查询: '{query}'\n")
print("相似度排名:")
for sim, sent in results:
    print(f"  {sim:.4f} | {sent}")

# --------- 验证 ---------
assert len(results) == len(sentences), "结果数量应与句子数量一致"
assert results[0][0] > results[-1][0], "排序应从高到低"
print("\n✅ 验证通过！")

# --------- 加分：聚类 ---------
# 取消下面的注释试试 KMeans 聚类
# from sklearn.cluster import KMeans
# kmeans = KMeans(n_clusters=2, random_state=42)
# labels = kmeans.fit_predict(np.array(vectors))
# print("\n聚类结果:")
# for label, sent in zip(labels, sentences):
#     print(f"  簇 {label}: {sent}")

---
## 第四部分：Prompt Engineering 速成（20 min）

### 核心问题：有了大模型，很多任务不需要训练，只需要写好提示词 (Prompt)！

三种基本策略：

| 策略 | 说明 | 适用场景 |
|------|------|----------|
| **Zero-shot** | 直接提问，不给示例 | 简单任务 |
| **Few-shot** | 给几个示例，再提问 | 需要特定格式/风格 |
| **Chain-of-Thought** | 要求模型逐步推理 | 复杂推理任务 |

In [ ]:
# 初始化 LLM 后端（使用 Day0 配置的后端和模型）
try:
    llm = env.get_llm()
    # 快速测试
    _test = llm.generate("hi", max_tokens=5)
    print("LLM 就绪！")
except Exception as e:
    print(f"LLM 初始化失败: {e}")
    print("请确认已运行 Day0 完成环境配置")
    llm = None

In [ ]:
# 策略 1: Zero-shot — 直接提问
if llm:
    prompt_zero = "请将以下客户反馈分类为 正面/负面/中性：\n\n'这个产品质量太差了，用了一周就坏了'"
    
    print("=" * 50)
    print("Zero-shot Prompt:")
    print(prompt_zero)
    print("=" * 50)
    
    response = llm.generate(prompt_zero, temperature=0)
    print(f"\n模型回复:\n{response}")
else:
    print("LLM 未初始化，请先运行上方单元格")
    print("\n预期回复: 负面")

In [ ]:
# 策略 2: Few-shot — 给示例
if llm:
    prompt_few = """请将客户反馈分类为 正面/负面/中性。

示例:
反馈: "发货很快，包装完好" → 正面
反馈: "产品有划痕，不太满意" → 负面
反馈: "还行吧，一般般" → 中性

请分类:
反馈: "客服态度非常好，问题很快解决了"
分类:"""
    
    print("=" * 50)
    print("Few-shot Prompt:")
    print(prompt_few)
    print("=" * 50)
    
    response = llm.generate(prompt_few, temperature=0)
    print(f"\n模型回复:\n{response}")
else:
    print("预期回复: 正面")

In [ ]:
# 策略 3: Chain-of-Thought (CoT) — 逐步推理
if llm:
    prompt_cot = """请分析以下客户反馈，判断情感类别。

反馈: "产品功能很强大，但是价格有点贵，总体来说还是值得购买的"

请按以下步骤分析:
1. 找出正面表述
2. 找出负面表述
3. 判断整体倾向
4. 给出最终分类（正面/负面/中性）"""
    
    print("=" * 50)
    print("Chain-of-Thought Prompt:")
    print(prompt_cot)
    print("=" * 50)
    
    response = llm.generate(prompt_cot, temperature=0)
    print(f"\n模型回复:\n{response}")
else:
    print("预期: 模型会逐步分析正面/负面因素，最终给出分类")

### Prompt 工程企业实践要点

| 技巧 | 说明 |
|------|------|
| 明确角色 | `"你是一个专业的客服分类系统"` |
| 限制输出格式 | `"只回复：正面/负面/中性"` |
| 给示例 | Few-shot 比 Zero-shot 更稳定 |
| 分步推理 | CoT 适合复杂判断 |
| 控制 temperature | 分类任务用 0，创作任务用 0.7+ |

---
## 🏋️ 练习 5：Prompt工程A/B测试（15分钟）⭐⭐

**目标**：设计两种不同的 Prompt 策略（Few-shot vs Zero-shot），在测试数据上做A/B测试，对比分类准确率

**步骤**：
1. Part A：设计一个 Few-shot prompt 模板 `PROMPT_A`，包含2-3个分类示例
2. Part B：设计一个 Zero-shot prompt 模板 `PROMPT_B`，只有指令无示例
3. 在8条测试投诉上分别用两个 prompt 调用 LLM，记录分类结果
4. 打印对比表格，计算两种策略的准确率

**分类类别**：产品质量 / 物流配送 / 售后服务 / 价格争议 / 账户问题 / 其他

**预期输出**：
```
测试文本              预期      Prompt A   Prompt B   A✓   B✓
收到的商品有明显划痕... 产品质量   产品质量    产品质量    ✓    ✓
...
准确率: Prompt A = x/8 (xx%), Prompt B = x/8 (xx%)
```

> 注意：此练习需要已配置 API Key，请确保已运行 Day0 环境配置。

<details>
<summary>💡 提示1：思路方向</summary>

Few-shot prompt 应包含：任务描述 + 2-3个示例（输入->输出）+ 待分类文本。Zero-shot 只需指令 + 待分类文本。

</details>

<details>
<summary>💡 提示2：关键代码</summary>

```python
full_prompt_a = PROMPT_A.format(text=text)
result_a = llm.generate(full_prompt_a, temperature=0)
a_correct = expected in result_a
```

</details>

<details>
<summary>💡 提示3：参考答案</summary>

```python
for text, expected in complaints:
    result_a = llm.generate(prompt_a.format(text=text), temperature=0)
    result_b = llm.generate(prompt_b.format(text=text), temperature=0)
    results.append({
        "text": text, "expected": expected,
        "result_a": result_a, "result_b": result_b,
        "a_correct": expected in result_a,
        "b_correct": expected in result_b,
    })
```

</details>

In [ ]:
# 练习 5：Prompt工程A/B测试

# ↓↓↓ 你的代码：设计两个 Prompt 模板 ↓↓↓

# Prompt A：Few-shot（带示例）
PROMPT_A = """你是一个客户投诉分类系统。请将客户投诉分类为以下类别之一：产品质量/物流配送/售后服务/价格争议/账户问题/其他

示例：
投诉: "收到的手机屏幕有裂痕" → 产品质量
投诉: "包裹一直没有送到" → 物流配送
投诉: "退款申请没人处理" → 售后服务

请分类以下投诉，只回复类别名称：
投诉: "{text}"
分类:"""

# Prompt B：Zero-shot（纯指令，无示例）
PROMPT_B = """请将以下客户投诉分类为：产品质量/物流配送/售后服务/价格争议/账户问题/其他。只回复类别名称。

投诉: "{text}"
分类:"""

# ↑↑↑ 你的代码 ↑↑↑

# 测试数据
test_complaints = [
    ("收到的商品有明显划痕，包装也破损了", "产品质量"),
    ("快递显示已签收但我没收到", "物流配送"),
    ("申请退款三天了还没处理", "售后服务"),
    ("同样的东西别家便宜很多", "价格争议"),
    ("我的账号突然登不上了", "账户问题"),
    ("产品用了两天就坏了", "产品质量"),
    ("发货太慢了等了一周", "物流配送"),
    ("客服态度很差不解决问题", "售后服务"),
]

# A/B 测试函数
def run_ab_test(complaints, prompt_a, prompt_b, llm):
    """对比两个 prompt 在测试集上的分类效果"""
    results = []
    if llm is None:
        print("LLM 未初始化，跳过 A/B 测试")
        print("请先运行 Day0 配置 API Key")
        return results
    for text, expected in complaints:
        # ↓↓↓ 你的代码（约6行）↓↓↓
        result_a = llm.generate(prompt_a.format(text=text), temperature=0)
        result_b = llm.generate(prompt_b.format(text=text), temperature=0)
        results.append({
            "text": text, "expected": expected,
            "result_a": result_a, "result_b": result_b,
            "a_correct": expected in result_a,
            "b_correct": expected in result_b,
        })
        # ↑↑↑ 你的代码 ↑↑↑
    return results

results = run_ab_test(test_complaints, PROMPT_A, PROMPT_B, llm)

# 打印对比表
if results:
    print(f"{'测试文本':<20} {'预期':^8} {'Prompt A':^10} {'Prompt B':^10} {'A✓':^4} {'B✓':^4}")
    print("-" * 60)
    correct_a = correct_b = 0
    for r in results:
        a_ok = "✓" if r['a_correct'] else "✗"
        b_ok = "✓" if r['b_correct'] else "✗"
        if r['a_correct']: correct_a += 1
        if r['b_correct']: correct_b += 1
        print(f"{r['text'][:18]:<20} {r['expected']:^8} {r['result_a'][:8]:^10} {r['result_b'][:8]:^10} {a_ok:^4} {b_ok:^4}")
    print("-" * 60)
    total = len(results)
    print(f"准确率: Prompt A = {correct_a}/{total} ({correct_a/total*100:.0f}%), Prompt B = {correct_b}/{total} ({correct_b/total*100:.0f}%)")
    winner = "Few-shot (A)" if correct_a >= correct_b else "Zero-shot (B)"
    print(f"\n结论: {winner} 效果更好")

---
## 第五部分：企业决策框架 — 微调 vs RAG vs Prompt（15 min）

### 三种方案对比

| 维度 | Prompt Engineering | RAG (检索增强生成) | 微调 (Fine-tuning) |
|------|-------------------|-------------------|--------------------|
| **成本** | 最低（API 费用） | 中等（向量数据库 + API） | 最高（GPU + 标注数据） |
| **开发周期** | 小时级 | 天级 | 周~月级 |
| **数据需求** | 无需训练数据 | 需要知识库文档 | 需要标注数据（通常 1000+） |
| **准确率** | 中等 | 中高（依赖知识库质量） | 最高 |
| **可定制性** | 低 | 中 | 高 |
| **维护成本** | 低 | 中（需更新知识库） | 高（需重新训练） |

### 决策流程图

```
你的任务是什么？
  ├─ 通用任务（翻译/摘要/对话）→ 直接用 Prompt Engineering
  ├─ 需要最新/专有知识？ → RAG
  │    └─ 知识库文档 < 100 页？ → 直接放进 context（长上下文模型）
  │    └─ 知识库文档 > 100 页？ → 搭建向量数据库 + RAG
  ├─ 需要特定风格/格式？ → Few-shot Prompt 先试
  │    └─ Few-shot 不够好？ → 考虑微调
  └─ 需要极高准确率 + 大量数据？ → 微调（SFT/LoRA）
```

### 企业快速决策清单

1. **先试 Prompt**：80% 的任务用好 prompt 就够了
2. **数据量少 (<100条)**：Few-shot > Fine-tuning
3. **需要实时知识**：RAG 是唯一选择
4. **对延迟敏感**：微调小模型 > 调用大模型 API
5. **预算有限**：Prompt > RAG > 微调

In [ ]:
# 企业场景决策练习
scenarios = [
    {
        "场景": "电商客服自动回复，需要查询订单信息和退换货政策",
        "推荐": "RAG",
        "原因": "需要实时查询订单数据 + 公司政策文档，Prompt 无法获取实时信息"
    },
    {
        "场景": "将客户邮件翻译成英文",
        "推荐": "Prompt Engineering",
        "原因": "翻译是通用能力，大模型已经做得很好，不需要额外训练"
    },
    {
        "场景": "医疗影像报告自动生成，需要符合特定医院的格式规范",
        "推荐": "微调 (Fine-tuning)",
        "原因": "需要极高准确率 + 特定格式 + 专业领域知识，且有大量历史报告可用于训练"
    },
    {
        "场景": "内部知识库问答，文档约 500 页",
        "推荐": "RAG",
        "原因": "文档量大无法全部放进 context，需要检索最相关的片段"
    },
]

print("企业 AI 场景决策示例")
print("=" * 60)
for s in scenarios:
    print(f"\n场景: {s['场景']}")
    print(f"推荐方案: {s['推荐']}")
    print(f"原因: {s['原因']}")
    print("-" * 60)

---
## 上午总结

### 今天上午我们走过了从文本到向量的完整路径

```
文本 → Tokenizer → Token IDs → Embedding → 向量 → 模型 → 输出
```

### 关键收获

| 概念 | 一句话总结 |
|------|----------|
| 训练循环 | Forward → Loss → Backward → Step，反复迭代 |
| Tokenizer | 把文本切成子词 (subword)，中文效率低于英文 |
| Embedding | 可学习的查表矩阵，把离散 ID 变成稠密向量 |
| Prompt Engineering | 80% 的任务用好 prompt 就够了 |
| 企业决策 | 先 Prompt → 不够就 RAG → 最后考虑微调 |

### 下午预告

下午我们将深入 **Transformer 架构**，理解让 GPT/Qwen/LLaMA 如此强大的核心机制 —— **Self-Attention（自注意力）**。